In [1]:
################################################################
#   LIBRARIES
################################################################

# Import necessary python libraries
import pandas as pd
import os
import sys
import numpy as np
import re

from IPython.display import display, Markdown
from dateutil.relativedelta import relativedelta
import hashlib

**HDB Technical Assessment**

**<u>Data Quality</u>**
1. Combine the datasets into a single master dataset using Jan 2012 dataset as the strict authoritative set.

In [2]:

# Define the base directory
base_dir = r"C:\Users\teoni\Documents\GitHub\housing_data\ResaleFlatPrices"

# List of CSV files to combine
csv_files = [
    "Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",
    "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
    "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
    "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
    "Resale flat prices based on registration date from Jan-2017 onwards.csv"
]

# Define the standard columns we want in the final dataset
standard_columns = [
    'month', 'town', 'flat_type', 'block', 'street_name',
    'storey_range', 'floor_area_sqm', 'flat_model',
    'lease_commence_date', 'remaining_lease', 'resale_price'
]

In [3]:
# Function to combine datasets
def combine_hdb_datasets():
    # List to hold dataframes
    dataframes = []

    for file in csv_files:
        file_path = os.path.join(base_dir, file)
        print(f"Processing {file}...")
        # Read the CSV file
        df = pd.read_csv(file_path)

        # 'remaining_lease' column is missing from certain files, add them in
        if 'remaining_lease' not in df.columns:
            # Add empty 'remaining_lease' column for older files
            df['remaining_lease'] = None

        # Ensure columns are in the standard order
        df = df[standard_columns]

        dataframes.append(df)
        print(f"  -> Added {len(df)} records")

    # Combine all dataframes
    master_df = pd.concat(dataframes, ignore_index=True)

    # Sort by month for better organization
    master_df = master_df.sort_values('month').reset_index(drop=True)

    # Save the master dataset
    output_path = os.path.join(base_dir, "HDB_Resale_Prices_Master_Dataset.csv")
    master_df.to_csv(output_path, index=False)

    print(f"\nMaster dataset created successfully!")
    print(f"Total records: {len(master_df)}")
    print(f"Saved to: {output_path}")

    return master_df

raw = combine_hdb_datasets()

Processing Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv...
  -> Added 287196 records
Processing Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv...
  -> Added 369651 records
Processing Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv...
  -> Added 52203 records
Processing Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv...
  -> Added 37153 records
Processing Resale flat prices based on registration date from Jan-2017 onwards.csv...
  -> Added 240807 records

Master dataset created successfully!
Total records: 987010
Saved to: C:\Users\teoni\Documents\GitHub\housing_data\ResaleFlatPrices\HDB_Resale_Prices_Master_Dataset.csv


**<u>Data Quality</u>**

2. Perform data profiling on the dataset

In [4]:
# Function to profile data on a pandas DataFrame

def profile_dataframe(df: pd.DataFrame) -> dict:
    """
    Perform comprehensive data profiling on a pandas DataFrame.

    Covers:
        - Structural profiling 
        - Completeness
        - Uniqueness
        - Validity
        - Distribution
        - Outliers
        - Pattern analysis

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.

    Returns
    -------
    dict
        Dictionary containing profiling results.
    """

    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    # ---------------------------------------------------------
    # 1. STRUCTURAL PROFILING
    # ---------------------------------------------------------

    structure = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "duplicate_row_count": int(df.duplicated().sum()),
        "memory_usage_mb": round(
            df.memory_usage(deep=True).sum() / 1024**2, 2
        ),
        "columns": list(df.columns),
    }

    # ---------------------------------------------------------
    # 2. COLUMN-LEVEL PROFILING
    # ---------------------------------------------------------

    column_profiles = []

    for column in df.columns:

        series = df[column]

        total = len(series)
        null_count = int(series.isna().sum())
        non_null_count = total - null_count

        # ---------------------------------------------
        # Basic metadata
        # ---------------------------------------------

        profile = {
            "column": column,
            "dtype": str(series.dtype),
            "total_count": total,
            "non_null_count": non_null_count,
            "null_count": null_count,
            "null_pct": round(
                (null_count / total * 100) if total else 0, 2
            ),
            "distinct_count": int(series.nunique(dropna=True)),
            "distinct_pct": round(
                (series.nunique(dropna=True) / total * 100)
                if total else 0,
                2
            ),
            "duplicate_count": int(series.duplicated().sum()),
            "unique": bool(series.nunique(dropna=True) == non_null_count),
        }

        # ---------------------------------------------
        # Completeness
        # ---------------------------------------------

        if series.dtype == "object":
            blank_count = int(
                series.fillna("").astype(str).str.strip().eq("").sum()
            )
        else:
            blank_count = 0

        profile["blank_count"] = blank_count
        profile["blank_pct"] = round(
            (blank_count / total * 100) if total else 0,
            2
        )

        # ---------------------------------------------
        # Numeric profiling
        # ---------------------------------------------

        if pd.api.types.is_numeric_dtype(series):

            profile.update({
                "min": series.min(),
                "max": series.max(),
                "mean": series.mean(),
                "median": series.median(),
                "std": series.std(),
                "q25": series.quantile(0.25),
                "q50": series.quantile(0.50),
                "q75": series.quantile(0.75),
            })

            # IQR-based outlier detection
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outliers = series[
                (series < lower_bound) |
                (series > upper_bound)
            ]

            profile["outlier_count"] = int(outliers.count())
            profile["outlier_pct"] = round(
                (outliers.count() / non_null_count * 100)
                if non_null_count else 0,
                2
            )

        # ---------------------------------------------
        # Date profiling
        # ---------------------------------------------

        elif pd.api.types.is_datetime64_any_dtype(series):

            profile.update({
                "min_date": series.min(),
                "max_date": series.max(),
                "future_date_count": int(
                    (series > pd.Timestamp.now()).sum()
                ),
            })

        # ---------------------------------------------
        # Text / categorical profiling
        # ---------------------------------------------

        elif pd.api.types.is_string_dtype(series) or \
                series.dtype == "object":

            non_null = series.dropna().astype(str)

            if len(non_null) > 0:

                lengths = non_null.str.len()

                profile.update({
                    "min_length": int(lengths.min()),
                    "max_length": int(lengths.max()),
                    "avg_length": round(lengths.mean(), 2),

                    # Most frequent values
                    "top_values": (
                        series.value_counts(dropna=True)
                        .head(5)
                        .to_dict()
                    ),
                })

                # -----------------------------------------
                # Pattern analysis
                # -----------------------------------------

                profile["email_pattern_count"] = int(
                    non_null.str.match(
                        r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
                    ).sum()
                )

                profile["numeric_only_count"] = int(
                    non_null.str.match(r"^\d+$").sum()
                )

                profile["alpha_only_count"] = int(
                    non_null.str.match(r"^[A-Za-z]+$").sum()
                )

                profile["alphanumeric_count"] = int(
                    non_null.str.match(r"^[A-Za-z0-9]+$").sum()
                )

        # ---------------------------------------------
        # Validity checks
        # ---------------------------------------------

        profile["negative_count"] = None

        if pd.api.types.is_numeric_dtype(series):
            profile["negative_count"] = int((series < 0).sum())

        column_profiles.append(profile)

    # ---------------------------------------------------------
    # 3. COLUMN PROFILE DATAFRAME
    # ---------------------------------------------------------

    column_profile_df = pd.DataFrame(column_profiles)

    # ---------------------------------------------------------
    # 4. DUPLICATE RECORD PROFILING
    # ---------------------------------------------------------

    duplicate_rows = df[df.duplicated(keep=False)].copy()

    # ---------------------------------------------------------
    # 5. CORRELATION PROFILING
    # ---------------------------------------------------------

    numeric_df = df.select_dtypes(include=np.number)

    if not numeric_df.empty:
        correlation_matrix = numeric_df.corr()
    else:
        correlation_matrix = pd.DataFrame()

    # ---------------------------------------------------------
    # RETURN RESULTS
    # ---------------------------------------------------------

    return {
        "structure": structure,
        "columns": column_profile_df,
        "duplicates": duplicate_rows,
        "correlations": correlation_matrix,
    }

def display_profile(profile):

    display(Markdown("# Data Profiling Report"))

    # Structure
    display(Markdown("## Dataset Structure"))

    structure_df = pd.DataFrame(
        list(profile["structure"].items()),
        columns=["Metric", "Value"]
    )

    display(structure_df)

    # Columns
    display(Markdown("## Column Profile"))
    display(profile["columns"])

    # Duplicates
    display(Markdown("## Duplicate Rows"))
    display(profile["duplicates"])

    # Correlations
    display(Markdown("## Correlations"))
    display(profile["correlations"])
    
raw_profiled = profile_dataframe(raw)
display_profile(raw_profiled)

# Data Profiling Report

## Dataset Structure

,Metric,Value
0,row_count,987010
1,column_count,11
2,duplicate_row_count,1929
3,memory_usage_mb,431.63
4,columns,"[month, town, flat_type, block, street_name, s..."


## Column Profile

,column,dtype,total_count,non_null_count,null_count,null_pct,distinct_count,distinct_pct,duplicate_count,unique,...,min,max,mean,median,std,q25,q50,q75,outlier_count,outlier_pct
0,month,object,987010,987010,0,0.00,441,0.04,986569,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,town,object,987010,987010,0,0.00,27,0.00,986983,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,flat_type,object,987010,987010,0,0.00,8,0.00,987002,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,block,object,987010,987010,0,0.00,2785,0.28,984225,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,street_name,object,987010,987010,0,0.00,597,0.06,986413,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,storey_range,object,987010,987010,0,0.00,25,0.00,986985,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,floor_area_sqm,float64,987010,987010,0,0.00,234,0.02,986776,False,...,28.0,366.7,95.667239,93.0,25.715294,73.0,93.0,113.0,2517.0,0.26
7,flat_model,object,987010,987010,0,0.00,34,0.00,986976,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,lease_commence_date,int64,987010,987010,0,0.00,58,0.01,986952,False,...,1966.0,2023.0,1988.892266,1987.0,11.261829,1981.0,1987.0,1996.0,9206.0,0.93
9,remaining_lease,object,987010,277960,709050,71.84,753,0.08,986256,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Duplicate Rows

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
70,1990-01,JURONG WEST,EXECUTIVE,407,JURONG WEST ST 42,10 TO 12,150.0,MAISONETTE,1985,None,160000.0
90,1990-01,JURONG WEST,EXECUTIVE,407,JURONG WEST ST 42,10 TO 12,150.0,MAISONETTE,1985,None,160000.0
102,1990-01,QUEENSTOWN,3 ROOM,2,GHIM MOH RD,04 TO 06,65.0,IMPROVED,1976,None,36000.0
103,1990-01,QUEENSTOWN,3 ROOM,2,GHIM MOH RD,10 TO 12,65.0,IMPROVED,1976,None,40000.0
104,1990-01,QUEENSTOWN,3 ROOM,2,GHIM MOH RD,10 TO 12,65.0,IMPROVED,1976,None,40000.0
...,...,...,...,...,...,...,...,...,...,...,...
980067,2026-06,PASIR RIS,4 ROOM,530A,PASIR RIS DR 1,07 TO 09,86.0,DBSS,2015,88 years,828000.0
983576,2026-08,JURONG WEST,4 ROOM,541,JURONG WEST AVE 1,04 TO 06,103.0,Model A,1984,57 years 04 months,500000.0
983578,2026-08,JURONG WEST,4 ROOM,541,JURONG WEST AVE 1,04 TO 06,103.0,Model A,1984,57 years 04 months,500000.0
983997,2026-08,CHOA CHU KANG,3 ROOM,10,TECK WHYE AVE,04 TO 06,67.0,New Generation,1978,50 years 09 months,330000.0


## Correlations

,floor_area_sqm,lease_commence_date,resale_price
floor_area_sqm,1.000000,0.346584,0.514699
lease_commence_date,0.346584,1.000000,0.554090
resale_price,0.514699,0.554090,1.000000


In [5]:
#drop NA and duplicate records
raw_cleaned = raw.drop_duplicates()
raw_cleaned = raw_cleaned.dropna(how='all')

**<u>Data Quality</u>**

3. Assume HDB lease is 99 years old and compute the remaining lease as of 1st Jan 2027, rounded down to
Years and Months.

In [6]:
as_of_date = pd.Timestamp("2027-01-01")

def calculate_remaining_lease(year):
    # convert lease commence date into a YYYY-MM-DD format
    lease_start = pd.Timestamp(f"{int(year)}-01-01")
    # add 99 years to the formatted lease start date
    lease_end = lease_start + pd.DateOffset(years=99)

    # compare lease end date with 2027-01-01 to find the difference in years and months
    difference = relativedelta(
        lease_end.to_pydatetime(),
        as_of_date.to_pydatetime()
    )
    return f"{difference.years} years {difference.months} months"

# add a column to represent the remaining lease per record
raw_cleaned["remaining_lease"] = raw_cleaned["lease_commence_date"].apply(calculate_remaining_lease)

**<u>Data Quality</u>**

4. Design Data Validation rules to validate Town, Flat Type, Flat Model, and storey_range using Jan 2012
dataset as the strict authoritative set. Any rows that fail the validation should be quarantined.

In [7]:
def data_validation(df):
    
    # Extract Jan 2012 subset as authoritative set
    jan_2012 = df[df['month'] == '2012-01']
    if jan_2012.empty:
        print("Warning: No data found for Jan 2012. Cannot create authoritative set.")
        return

    # Get unique values for each column in the authoritative set
    authoritative_towns = set(jan_2012['town'].dropna().unique())
    authoritative_flat_types = set(jan_2012['flat_type'].dropna().unique())
    authoritative_flat_models = set(jan_2012['flat_model'].dropna().unique())
    authoritative_storey_ranges = set(jan_2012['storey_range'].dropna().unique())

    valid_town = df['town'].isin(authoritative_towns)
    valid_flat_type = df['flat_type'].isin(authoritative_flat_types)
    valid_flat_model = df['flat_model'].isin(authoritative_flat_models)
    valid_storey_range = df['storey_range'].isin(authoritative_storey_ranges)

    # Combine masks: a row is valid only if all four are True
    valid_mask = valid_town & valid_flat_type & valid_flat_model & valid_storey_range

    # Split into valid and quarantined
    valid_df = df[valid_mask].copy()
    quarantined_df = df[~valid_mask].copy()

    return valid_df, quarantined_df

raw_cleaned, raw_quarantined = data_validation(raw_cleaned)

In [8]:
print(raw_cleaned.head())
print(raw_quarantined.head())

          month        town flat_type block  street_name storey_range  \
287196  2000-01  QUEENSTOWN    3 ROOM    41   HOLLAND DR     04 TO 06   
287197  2000-01  QUEENSTOWN    3 ROOM     5   DOVER CRES     07 TO 09   
287198  2000-01  QUEENSTOWN    3 ROOM     5   DOVER CRES     04 TO 06   
287199  2000-01  QUEENSTOWN    3 ROOM     5   DOVER CRES     10 TO 12   
287200  2000-01  QUEENSTOWN    3 ROOM   166  STIRLING RD     01 TO 03   

        floor_area_sqm      flat_model  lease_commence_date  \
287196            59.0        Improved                 1975   
287197            67.0  New Generation                 1979   
287198            82.0  New Generation                 1979   
287199            67.0  New Generation                 1979   
287200            60.0        Improved                 1970   

          remaining_lease  resale_price  
287196  47 years 0 months      155000.0  
287197  51 years 0 months      158000.0  
287198  51 years 0 months      182000.0  
287199  51 yea

**<u>Data Quality</u>**

5. Identify potentially anomalous resale price using appropriate heuristics.

In [9]:
def identify_price_anomalies(df):
    df = df.copy()

    group_cols = ["town", "flat_type"]

    # Calculate Q1, Q3 and IQR within each group
    q1 = df.groupby(group_cols)["resale_price"].transform("quantile", 0.25)
    q3 = df.groupby(group_cols)["resale_price"].transform("quantile", 0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Flag anomalies
    df["price_anomaly"] = (
        (df["resale_price"] < lower_bound) |
        (df["resale_price"] > upper_bound)
    )

    # Store reason
    df["price_anomaly_type"] = "Normal"

    df.loc[
        df["resale_price"] > upper_bound,
        "price_anomaly_type"
    ] = "Unusually High"

    df.loc[
        df["resale_price"] < lower_bound,
        "price_anomaly_type"
    ] = "Unusually Low"

    return df

In [10]:
cleaned = identify_price_anomalies(raw_cleaned)

cleaned = cleaned[
    cleaned["price_anomaly_type"] == "Normal"
]

print(cleaned)

          month         town  flat_type block        street_name storey_range  \
287196  2000-01   QUEENSTOWN     3 ROOM    41         HOLLAND DR     04 TO 06   
287197  2000-01   QUEENSTOWN     3 ROOM     5         DOVER CRES     07 TO 09   
287198  2000-01   QUEENSTOWN     3 ROOM     5         DOVER CRES     04 TO 06   
287199  2000-01   QUEENSTOWN     3 ROOM     5         DOVER CRES     10 TO 12   
287200  2000-01   QUEENSTOWN     3 ROOM   166        STIRLING RD     01 TO 03   
...         ...          ...        ...   ...                ...          ...   
987005  2026-09   QUEENSTOWN     5 ROOM    26         DOVER CRES     16 TO 18   
987006  2026-09      PUNGGOL     4 ROOM  303D         PUNGGOL PL     07 TO 09   
987007  2026-09      PUNGGOL     4 ROOM  274A         PUNGGOL PL     13 TO 15   
987008  2026-09   QUEENSTOWN     5 ROOM    20         HOLLAND DR     19 TO 21   
987009  2026-09  JURONG WEST  EXECUTIVE   912  JURONG WEST ST 91     01 TO 03   

        floor_area_sqm     

**<u>Data Transformation</u>**

4. Create Resale Identifier

In [11]:
def generate_resale_identifier(df):
    """
    Add a 'Resale Identifier' column to the dataframe based on specified rules.
 
    Parameters:
    df (pd.DataFrame): Input dataframe with columns:
        - block: str or int, block number (may contain non-numeric characters)
        - resale_price: numeric, resale price
        - town: str, town name
        - month: str or datetime, in format 'YYYY-MM' or similar
        - flat_type: str, flat type (e.g., 'EXECUTIVE', '2 ROOM', etc.)
 
    Returns:
    pd.DataFrame: Dataframe with new 'Resale Identifier' column.
    """
    # Make a copy to avoid modifying original
    df = df.copy()
 
    # 1. Process block column: extract digits, pad to 3 with leading zeros
    def process_block(val):
        # Convert to string and extract digits
        digits = ''.join(filter(str.isdigit, str(val)))
        # Take first 3 digits, if less than 3 pad with zeros on left
        if len(digits) >= 3:
            return digits[:3].zfill(3)
        else:
            return digits.zfill(3)
 
    df['block_digits'] = df['block'].apply(process_block)
 
    # 2. Compute average resale price per group (year-month, town, flat_type)
    # Ensure month is datetime for extraction
    if not pd.api.types.is_datetime64_any_dtype(df['month']):
        df['month_dt'] = pd.to_datetime(df['month'], errors='coerce')
    else:
        df['month_dt'] = df['month']
 
    # Create year-month key for grouping (using full year-month for averaging)
    df['year_month_key'] = df['month_dt'].dt.to_period('M')
 
    # Compute average resale price per group
    avg_price = df.groupby(['year_month_key', 'town', 'flat_type'])['resale_price'].transform('mean')
    # Take integer part (round) and get first 3 digits
    def get_avg_digits(val):
        # Round to nearest integer
        int_val = int(round(val))
        # Convert to string and take first 3 digits, pad with zeros if needed
        s = str(int_val)
        if len(s) >= 3:
            return s[:3]
        else:
            return s.zfill(3)
 
    df['avg_digits'] = avg_price.apply(get_avg_digits)
 
    # 3. Last four digits: year-month as YYMM (last two digits of year + month)
    def get_ym_ymd(dt):
        if pd.isnull(dt):
            return '0000'
        year_last_two = dt.year % 100
        month = dt.month
        return f"{year_last_two:02d}{month:02d}"
 
    df['ym_digits'] = df['month_dt'].apply(get_ym_ymd)
 
    # 4. First character of town
    df['town_first'] = df['town'].astype(str).str[0].str.upper()
 
    # 5. First character is always 'S'
    df['resale_identifier'] = (
        'S' +
        df['block_digits'] +
        df['avg_digits'] +
        df['ym_digits'] +
        df['town_first']
    )

    return df

def duplicate_identification_algorithm(df): 
    # Check for duplicates
    dup_mask = df.duplicated(subset=['resale_identifier'], keep=False)
    if dup_mask.any():
        # For duplicates, append a hash of the full row to make unique
        def row_hash(row):
            # Create a string representation of the row (excluding the new identifier columns)
            # Use all original columns for hashing
            row_str = ''.join([str(row[col]) for col in df.columns if col not in [
                'block_digits', 'avg_digits', 'ym_digits', 'town_first', 'resale_identifier',
                'month_dt', 'year_month_key'  # temporary columns
            ]])
            # Return first 4 characters of MD5 hash
            return hashlib.md5(row_str.encode()).hexdigest()[:4]
 
        # Apply hash only to duplicated rows, but we can apply to all for simplicity
        df['hash_suffix'] = df.apply(row_hash, axis=1)
        df['resale_identifier'] = df['resale_identifier'] + df['hash_suffix']
        # Drop helper columns
        df = df.drop(columns=['hash_suffix'])
 
    # Drop temporary columns used for computation
    df = df.drop(columns=['block_digits', 'avg_digits', 'ym_digits', 'town_first', 'month_dt', 'year_month_key'])
 
    return df

In [12]:
transformed = generate_resale_identifier(cleaned)
hashed = duplicate_identification_algorithm(transformed)